In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 128
block_size = 128 #es context length
eval_interval = 100
learning_rate = 3e-4
eval_iters = 200
n_embd = 512
n_head = 8 #used 8 masked self attention modules
n_layer = 6 #using 6 number of decoders
dropout = 0.2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

In [ ]:
# data
with open("../TrainingData/pg10.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [3]:
# letter level tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

In [4]:
# train/val split
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [5]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [6]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.mean().item()  # ← add .mean()
        out[split] = losses.mean()
    model.train()
    return out

In [7]:
def get_lr(it):
    if it < 100:
        return learning_rate * it / 100
    decay_ratio = (it - 100) / (max_iters - 100)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return learning_rate * 0.1 + coeff * learning_rate * 0.9

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        out = F.scaled_dot_product_attention(q, k, v, 
              attn_mask=None, 
              dropout_p=dropout if self.training else 0.0, 
              is_causal=True)
        return out

In [ ]:

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

In [ ]:

class FeedForward(nn.Module):
    """ feed-forward block """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:

class Block(nn.Module):
    """ transformer block """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [8]:

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = self.blocks(tok_emb + pos_emb)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [9]:
# init model
model = GPTLanguageModel().to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)
m = model.module if isinstance(model, torch.nn.DataParallel) else model
print(f"{sum(p.numel() for p in m.parameters())/1e6:.2f}M parameters")

# optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

Using 2 GPUs
19.05M parameters


In [ ]:
import os

# ── resume if checkpoint exists ──────────────────────────────────────────
checkpoint_path = '../model/bible_checkpoint.pt'
best_path       = '../model/bible_best.pt'

best_val_loss   = float('inf')
last_improvement = 0
start_iter      = 0

if os.path.exists(checkpoint_path):
    print("Found checkpoint, resuming...")
    ckpt = torch.load(checkpoint_path, map_location=device)
    m.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_iter       = ckpt['iter'] + 1
    best_val_loss    = ckpt['best_val_loss']
    last_improvement = ckpt['last_improvement']
    print(f"Resumed from step {start_iter}, best val loss {best_val_loss:.4f}")
else:
    print("No checkpoint found, starting fresh")

# ── training loop ────────────────────────────────────────────────────────
max_iters = 10000
patience  = 500

for iter in range(start_iter, max_iters):
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        # always save resumable checkpoint
        torch.save({
            'model_state_dict':     m.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'vocab_size':           vocab_size,
            'stoi':                 stoi,
            'itos':                 itos,
            'iter':                 iter,
            'best_val_loss':        best_val_loss,
            'last_improvement':     last_improvement,
        }, checkpoint_path)

        # save best model separately
        if losses['val'] < best_val_loss:
            best_val_loss    = losses['val']
            last_improvement = iter
            torch.save({
                'model_state_dict': m.state_dict(),
                'vocab_size':       vocab_size,
                'stoi':             stoi,
                'itos':             itos,
            }, best_path)
            print(f"  ✓ best model saved (val loss {best_val_loss:.4f})")

        elif iter - last_improvement > patience:
            print(f"Early stopping at step {iter}")
            break

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    loss = loss.mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

No checkpoint found, starting fresh


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


step 0: train loss 4.4609, val loss 4.4688
  ✓ best model saved (val loss 4.4688)
step 100: train loss 2.2911, val loss 2.3708
  ✓ best model saved (val loss 2.3708)
step 200: train loss 1.8878, val loss 2.0131
  ✓ best model saved (val loss 2.0131)
step 300: train loss 1.5861, val loss 1.7465
  ✓ best model saved (val loss 1.7465)
step 400: train loss 1.4425, val loss 1.6327
  ✓ best model saved (val loss 1.6327)
step 500: train loss 1.3456, val loss 1.5423
  ✓ best model saved (val loss 1.5423)
step 600: train loss 1.2911, val loss 1.5033
  ✓ best model saved (val loss 1.5033)
step 700: train loss 1.2412, val loss 1.4578
  ✓ best model saved (val loss 1.4578)
step 800: train loss 1.2007, val loss 1.4246
  ✓ best model saved (val loss 1.4246)
step 900: train loss 1.1737, val loss 1.4069
  ✓ best model saved (val loss 1.4069)
step 1000: train loss 1.1490, val loss 1.3817
  ✓ best model saved (val loss 1.3817)
step 1100: train loss 1.1289, val loss 1.3650
  ✓ best model saved (val loss 

In [ ]:
def generate_text(prompt, max_new_tokens=500):
    encoded = encode(prompt)
    context = torch.tensor([encoded], dtype=torch.long, device=device)
    output = decode(m.generate(context, max_new_tokens=max_new_tokens)[0].tolist())
    print(f"Prompt: {prompt}\n")
    print(output)

In [ ]:
generate_text("And God said")